In [ ]:
import pandas as pd
from pathlib import Path
from pyproj import Transformer
folder = Path("FixedBikes")
df_fixed_bikes = {}

transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)
for item in folder.iterdir():
    if item.is_file():
        df = pd.read_excel(item)
        
        col_check = ['kl.00-01', 'kl.01-02', 'kl.02-03', 'kl.03-04', 'kl.04-05', 'kl.05-06',
       'kl.06-07', 'kl.07-08', 'kl.08-09', 'kl.09-10', 'kl.10-11', 'kl.11-12',
       'kl.12-13', 'kl.13-14', 'kl.14-15', 'kl.15-16', 'kl.16-17', 'kl.17-18',
       'kl.18-19', 'kl.19-20', 'kl.20-21', 'kl.21-22', 'kl.22-23', 'kl.23-24']
            
        df.drop(columns=["Spor"])
        df = df[~(df[col_check] == 0).all(axis=1)]
        hour_cols = [col for col in df.columns if col.startswith("kl.")]
        id_cols = [col for col in df.columns if col not in hour_cols]
        
        df[hour_cols] = df[hour_cols].fillna(0)
        # 2. Unpivot (melt) the hourly columns into rows
        df_long = df.melt(
            id_vars=id_cols,
            value_vars=hour_cols,
            var_name="hour_range",
            value_name="bike_count",
        )
        
        # 3. Combine 'Dato' and the start hour from 'kl.XX-YY' into a datetime column
        dates = pd.to_datetime(df_long["Dato"], format="%d.%m.%Y")
        hours = df_long["hour_range"].str.extract(r"kl\.(\d{2})")[0].astype(int)
        
        df_long["timestamp"] = dates + pd.to_timedelta(hours, unit="h")
        
        # 4. Drop helper columns and sort chronologically
        df_long = (
            df_long.drop(columns=["Dato", "hour_range"])
            .sort_values(["Vej-Id", "Spor", "timestamp"])
            .reset_index(drop=True)
        )

        # 2. Convert entire Pandas columns at once (vectorized)
        df_long["Xkoordinat"], df_long["Ykoordinat"] = transformer.transform(
            df_long["Xkoordinat"].to_numpy(),
            df_long["Ykoordinat"].to_numpy(),
        )
        df_long["hour"] = df_long["timestamp"].dt.hour
        name = (item.name).split("cykeltaellinger-")
        df_fixed_bikes[name[1][0:3]] = df_long

In [42]:
for key in df_fixed_bikes:
    print(key)

2011.xlsx
2013.xlsx
2005.xlsx
2008-1.xlsx
2006.xlsx
2009.xlsx
2010.xlsx
2007.xlsx
2012.xlsx


# Example data for 2013

In [35]:
df_fixed_bikes[0].head()

,Vej-Id,Vejnavn,Spor,Xkoordinat,Ykoordinat,bike_count,timestamp
0,101 1011358-0 0/ 15 +1,Dr. Louises Bro ( ml. Nørrebrogade og Frederik...,1.0,12.564408,55.686473,0.0,2011-06-02 00:00:00
1,101 1011358-0 0/ 15 +1,Dr. Louises Bro ( ml. Nørrebrogade og Frederik...,1.0,12.564408,55.686473,0.0,2011-06-02 01:00:00
2,101 1011358-0 0/ 15 +1,Dr. Louises Bro ( ml. Nørrebrogade og Frederik...,1.0,12.564408,55.686473,0.0,2011-06-02 02:00:00
3,101 1011358-0 0/ 15 +1,Dr. Louises Bro ( ml. Nørrebrogade og Frederik...,1.0,12.564408,55.686473,0.0,2011-06-02 03:00:00
4,101 1011358-0 0/ 15 +1,Dr. Louises Bro ( ml. Nørrebrogade og Frederik...,1.0,12.564408,55.686473,28.0,2011-06-02 04:00:00


In [38]:
# Group by hour to see peak times
hourly_counts = df_fixed_bikes[0].groupby('hour')['bike_count'].sum().reset_index()
print("Hourly bike count totals:")
display(hourly_counts)

Hourly bike count totals:


,hour,bike_count
0,0,578058.0
1,1,401199.0
2,2,320136.0
3,3,264071.0
4,4,234353.0
5,5,320867.0
6,6,920454.0
7,7,2606295.0
8,8,3320948.0
9,9,1955878.0
